## 三表关联 

In [56]:
import pandas as pd 
from pathlib import Path 


In [57]:
project_root = Path.cwd().parent
project_root
data_path = project_root.joinpath('data', 'raw') 
data_path

WindowsPath('d:/各类文件/项目/olist-delivery-risk/data/raw')

### 这里涉及到.ipynb和.py文件读取路径的差异
- `.ipynb`没有`__file__`因而使用`Path.cwd()`到当前`notebook`所在的目录

- `.py`有`__file__`因而使用`Path(__file__)`查到当前的`.py`文件的相对路径，一般用`.resolve()`变成绝对路径，后`.parent`等的操作

In [58]:
orders = pd.read_csv(data_path.joinpath('olist_orders_dataset.csv'))

# 我们这个项目最终需要预测：一笔订单是否会延迟送达，而不是某一条商品明细是否延迟。
# 因此需要用到items的聚合
items = pd.read_csv(project_root.joinpath('data', 'processed', 'order_items_agg.csv')) 

customers = pd.read_csv(data_path.joinpath('olist_customers_dataset.csv')) 



In [59]:
# 为了merge不增加行数，先检查 

print(orders.shape) 
print(items.shape) 
print(customers.shape)

print(orders['order_id'].is_unique) 

print(items['order_id'].is_unique) 

print(customers['customer_id'].is_unique) 
 


(99441, 8)
(98666, 5)
(99441, 5)
True
True
True


- 这里99441-98666=775,有order但是没有order_items

In [60]:
# merge合并 

orders_items = orders.merge(
    items, 
    on='order_id', 
    how='left'
) 
# 若不用how='left', 默认how='inner'
print(orders.shape) 
print(items.shape) 
print(orders_items.shape) 
print(orders_items['order_id'].is_unique) 

(99441, 8)
(98666, 5)
(99441, 12)
True


In [61]:
orders_items.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
item_count                        775
product_amount                    775
freight_amount                    775
total_amount                      775
dtype: int64

In [62]:
orders_items[
    orders_items['item_count'].isna()
]['order_status'].value_counts() 


order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

### 未匹配商品明细订单分析

在将订单表 `orders` 与商品明细聚合表 `order_items_agg` 进行左连接后，发现有 **775 个订单没有对应的商品明细信息**。

这些订单的商品相关字段：

- `item_count`
- `product_amount`
- `freight_amount`
- `total_amount`

均为空，说明这些订单不是部分商品信息缺失，而是整个订单没有进入商品明细记录。

进一步分析这些订单的订单状态：

| order_status | 数量 |
| :--- | ---: |
| unavailable | 603 |
| canceled | 164 |
| created | 5 |
| invoiced | 2 |
| shipped | 1 |

#### 结果分析

1. **unavailable 状态（603 条）**

   占未匹配订单的大多数。

   该状态表示商品不可用，订单可能在商品确认阶段失败，没有进入正常的商品履约流程，因此订单表中存在记录，但没有对应商品明细。

2. **canceled 状态（164 条）**

   该类订单已经取消，通常未完成完整交易流程，因此可能没有生成商品明细记录。

3. **created 状态（5 条）**

   该状态表示订单处于创建阶段。少量订单可能已经生成订单记录，但商品明细尚未同步或交易流程未继续。

4. **invoiced 状态（2 条）**

   该状态表示订单已经进入开票阶段。理论上应该存在商品信息，但少量缺失可能由数据同步异常导致。

5. **shipped 状态（1 条）**

   已发货订单通常应包含商品明细，该记录可能属于数据异常情况，但数量极少，对整体分析影响有限。

#### 总结

这 775 个缺失商品信息的订单主要集中在 `unavailable` 和 `canceled` 两类状态，占比超过 98%。

因此，该缺失现象主要由订单未完成正常履约流程导致，而不是随机的数据缺失。

在后续分析中保留这些订单，并继续采用左连接方式关联其他信息，以保证订单级数据完整性。

In [63]:
# 合并customer

print(customers.isna().sum()) 

orders_full = orders_items.merge(
    customers, 
    on='customer_id', 
    how='left'  
)
orders_full.head()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,item_count,product_amount,freight_amount,total_amount,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,29.99,8.72,38.71,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,118.70,22.76,141.46,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,159.90,19.22,179.12,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,45.00,27.20,72.20,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,19.90,8.72,28.62,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


In [64]:
print(orders.shape) 
print(orders_items.shape) 
print(orders_full.shape)

print(orders_full['order_id'].is_unique) 

print(orders_full['customer_city'].isna().sum()) 
print(orders_full['customer_state'].isna().sum()) 

(99441, 8)
(99441, 12)
(99441, 16)
True
0
0


In [65]:
# 保存最终的订单级数据 
processed_path = project_root.joinpath('data', 'processed') 

output_path = processed_path.joinpath('orders_full.csv') 

orders_full.to_csv(output_path, index=False) 


In [66]:
# 最后的merge检查 
print(orders_full.shape) 
print(orders.shape) 
print(orders_full.duplicated().sum()) 
print(orders_full['order_id'].is_unique) 
print(orders_full[['customer_city', 'customer_state']].isna().sum()) 
print(orders_full.columns)

(99441, 16)
(99441, 8)
0
True
customer_city     0
customer_state    0
dtype: int64
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'item_count', 'product_amount', 'freight_amount', 'total_amount',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state'],
      dtype='str')


## 最终订单级数据集检查

经过订单表（`orders`）、商品明细聚合表（`order_items_agg`）以及客户信息表（`customers`）关联后，得到最终订单级数据集 `orders_full`。

该数据集用于后续订单延迟分析和预测建模。

---

### 1. 数据规模

最终数据规模：

| 数据集 | 行数 | 列数 |
| --- | ---: | ---: |
| orders_full | 99441 | 16 |

数据粒度：

> 一行代表一个订单。

经过商品明细聚合后，商品信息已经转换为订单级特征，包括：

- 商品数量（`item_count`）
- 商品总金额（`product_amount`）
- 运费金额（`freight_amount`）
- 订单总金额（`total_amount`）

同时加入客户信息：

- 客户城市（`customer_city`）
- 客户州（`customer_state`）

---

### 2. 订单唯一性检查

检查订单编号是否唯一：

```python
orders_full['order_id'].is_unique

In [67]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders_full[date_cols] = orders_full[date_cols].apply(
    pd.to_datetime,
    errors="coerce"
)

In [68]:
# 1. 获批早于下单
bad_approval = (
    orders_full["order_approved_at"]
    < orders_full["order_purchase_timestamp"]
)

# 2. 交付物流早于获批
bad_carrier = (
    orders_full["order_delivered_carrier_date"]
    < orders_full["order_approved_at"]
)

# 3. 实际送达早于交付物流
bad_delivery = (
    orders_full["order_delivered_customer_date"]
    < orders_full["order_delivered_carrier_date"]
)


print("获批早于下单：", bad_approval.sum())
print("物流交接早于获批：", bad_carrier.sum())
print("实际送达早于物流交接：", bad_delivery.sum())


获批早于下单： 0
物流交接早于获批： 1359
实际送达早于物流交接： 23


In [69]:
amount_cols = [
    "item_count",
    "product_amount",
    "freight_amount",
    "total_amount"
]

print(
    orders_full[amount_cols].describe(
        percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]
    )
)

         item_count  product_amount  freight_amount  total_amount
count  98666.000000    98666.000000    98666.000000  98666.000000
mean       1.141731      137.754076       22.823562    160.577638
std        0.538452      210.645145       21.650909    220.466087
min        1.000000        0.850000        0.000000      9.590000
25%        1.000000       45.900000       13.850000     61.980000
50%        1.000000       86.900000       17.170000    105.290000
75%        1.000000      149.900000       24.040000    176.870000
95%        2.000000      399.900000       54.960000    450.530000
99%        3.000000      998.900000      104.983500   1063.321000
max       21.000000    13440.000000     1794.960000  13664.080000


In [70]:
# 1. 物流交接早于获批
abnormal_carrier = orders_full.loc[bad_carrier].copy()

abnormal_carrier["diff_hours"] = (
    abnormal_carrier["order_approved_at"]
    - abnormal_carrier["order_delivered_carrier_date"]
).dt.total_seconds() / 3600

print("物流交接早于获批：")
print(abnormal_carrier["diff_hours"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]
))

# 2. 实际送达早于物流交接
abnormal_delivery = orders_full.loc[bad_delivery].copy()

abnormal_delivery["diff_hours"] = (
    abnormal_delivery["order_delivered_carrier_date"]
    - abnormal_delivery["order_delivered_customer_date"]
).dt.total_seconds() / 3600

print("\n实际送达早于物流交接：")
print(abnormal_delivery["diff_hours"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]
))


物流交接早于获批：
count    1359.000000
mean       24.751987
std       115.307793
min         0.005833
25%         1.415417
50%        17.167778
75%        25.956667
95%        86.840778
99%       165.390439
max      4109.256111
Name: diff_hours, dtype: float64

实际送达早于物流交接：
count     23.000000
mean      78.455133
std       89.308070
min        0.388333
25%       23.338333
50%       39.864444
75%      128.759028
95%      187.382528
99%      342.921183
max      386.308056
Name: diff_hours, dtype: float64


In [71]:
for name, df in [
    ("物流交接早于获批", abnormal_carrier),
    ("实际送达早于物流交接", abnormal_delivery)
]:
    gap = df["diff_hours"]

    print(f"\n{name}")
    print("不超过1小时：", (gap <= 1).sum())
    print("1到24小时：", ((gap > 1) & (gap <= 24)).sum())
    print("超过24小时：", (gap > 24).sum())


物流交接早于获批
不超过1小时： 242
1到24小时： 659
超过24小时： 458

实际送达早于物流交接
不超过1小时： 2
1到24小时： 5
超过24小时： 16


In [72]:
print(
    abnormal_carrier[
        [
            "order_id",
            "order_status",
            "order_approved_at",
            "order_delivered_carrier_date",
            "diff_hours"
        ]
    ]
    .sort_values("diff_hours", ascending=False)
    .head(10)
)

                               order_id order_status   order_approved_at  \
25883  7c48bb55e8e4f7e56d412e9653db37bc    delivered 2018-07-16 18:50:22   
14562  1fab4ac9d85079b3da72a11475ae1685    delivered 2017-09-13 22:06:11   
46163  0184d4ddb259e1a4cfc2871888cf97b8    delivered 2017-09-13 22:17:15   
98710  1378f9601350615613cc8832d6789c5d    delivered 2017-09-13 22:03:51   
41592  8554cb37f7158cb0b082a841d24a4589    delivered 2017-09-13 21:58:04   
11738  cf72398d0690f841271b695bbfda82d2    delivered 2017-09-13 22:04:39   
85393  40de47dfa620d667117e4a6067b6e1ec    delivered 2017-09-13 21:58:38   
31211  bc4854efd86d9f42140c951c595d20c1    delivered 2017-09-13 22:00:51   
55302  77ca435b03fbf991e5027e3776e37885    delivered 2017-09-13 21:56:08   
68315  580f3268bc2075c5961b2d2929c7a35b    delivered 2017-09-13 22:03:42   

      order_delivered_carrier_date   diff_hours  
25883          2018-01-26 13:35:00  4109.256111  
14562          2017-09-04 13:10:23   224.930000  
46163        

In [73]:
# 实际送达早于获批
bad_approval_delivery = (
    orders_full["order_delivered_customer_date"]
    < orders_full["order_approved_at"]
)

# 两种异常可能重叠
print("物流早于获批：", bad_carrier.sum())
print("送达早于物流：", bad_delivery.sum())

print(
    "实际送达早于获批：",
    bad_approval_delivery.sum()
)

print(
    "物流早于获批 且 送达早于获批：",
    (bad_carrier & bad_approval_delivery).sum()
)

print(
    "两种原有异常重叠数量：",
    (bad_carrier & bad_delivery).sum()
)

物流早于获批： 1359
送达早于物流： 23
实际送达早于获批： 61
物流早于获批 且 送达早于获批： 61
两种原有异常重叠数量： 0


In [74]:
# 建模样本条件
valid_model = (
    orders_full["order_status"].eq("delivered")
    & orders_full["order_approved_at"].notna()
    & orders_full["order_delivered_customer_date"].notna()
    & orders_full["order_estimated_delivery_date"].notna()
)

# 实际需要从建模数据排除的订单
exclude_mask = (
    valid_model & bad_approval_delivery
)

print("61条异常中的订单状态：")
print(
    orders_full.loc[
        bad_approval_delivery, "order_status"
    ].value_counts()
)

print("\n符合建模条件的异常订单数：")
print(exclude_mask.sum())

# 查看这些订单实际送达比获批早多久
problem_orders = orders_full.loc[
    exclude_mask
].copy()

problem_orders["diff_hours"] = (
    problem_orders["order_approved_at"]
    - problem_orders["order_delivered_customer_date"]
).dt.total_seconds() / 3600

print("\n时间差统计：")
print(problem_orders["diff_hours"].describe())

print("\n时间差最大的10笔订单：")
display(
    problem_orders[
        [
            "order_id",
            "order_approved_at",
            "order_delivered_customer_date",
            "diff_hours"
        ]
    ].sort_values(
        "diff_hours", ascending=False
    ).head(10)
)

61条异常中的订单状态：
order_status
delivered    61
Name: count, dtype: int64

符合建模条件的异常订单数：
61

时间差统计：
count     61.000000
mean      39.054408
std       42.976017
min        0.088611
25%        3.452222
50%       24.823333
75%       55.826944
max      167.751667
Name: diff_hours, dtype: float64

时间差最大的10笔订单：


,order_id,order_approved_at,order_delivered_customer_date,diff_hours
31211,bc4854efd86d9f42140c951c595d20c1,2017-09-13 22:00:51,2017-09-06 22:15:45,167.751667
14562,1fab4ac9d85079b3da72a11475ae1685,2017-09-13 22:06:11,2017-09-08 20:13:03,121.885556
41592,8554cb37f7158cb0b082a841d24a4589,2017-09-13 21:58:04,2017-09-08 20:07:45,121.838611
85393,40de47dfa620d667117e4a6067b6e1ec,2017-09-13 21:58:38,2017-09-08 20:41:16,121.289444
91801,4387477eec4b3c89b39f3f454940d059,2018-08-20 15:56:29,2018-08-15 14:56:43,120.996111
50623,e73fe43cdcd166f7f0c6e3c2bf11a917,2018-08-20 15:57:28,2018-08-15 15:46:38,120.180556
61544,6dcf0aeb8b1eb4021c26e1d0e9394979,2018-08-20 15:59:18,2018-08-15 19:06:29,116.880278
25411,6b80bb20190715d71c43efff617bd659,2017-03-01 10:51:46,2017-02-24 14:27:26,116.405556
46163,0184d4ddb259e1a4cfc2871888cf97b8,2017-09-13 22:17:15,2017-09-09 15:12:44,103.075278
87076,bc1b85147b5edbb7cbefcf5c1bd5ded9,2018-07-29 23:30:54,2018-07-25 20:56:35,98.571944


In [75]:
import numpy as np

# 1. 检查非法金额及数量
print("商品数量小于等于0：",
      (orders_full["item_count"] <= 0).sum())

print("商品金额为负：",
      (orders_full["product_amount"] < 0).sum())

print("运费为负：",
      (orders_full["freight_amount"] < 0).sum())

print("订单总金额为负：",
      (orders_full["total_amount"] < 0).sum())

# 2. 检查金额计算是否一致
valid_amount = orders_full[
    ["product_amount", "freight_amount", "total_amount"]
].notna().all(axis=1)

bad_amount = valid_amount & ~np.isclose(
    orders_full["total_amount"],
    orders_full["product_amount"] + orders_full["freight_amount"],
    atol=0.01,
    rtol=0
)

print("金额计算不一致的订单数：",
      bad_amount.sum())

商品数量小于等于0： 0
商品金额为负： 0
运费为负： 0
订单总金额为负： 0
金额计算不一致的订单数： 0


In [76]:
#高金额商品
high_amount = orders_full[
    [
        "order_id",
        "order_status",
        "item_count",
        "product_amount",
        "freight_amount",
        "total_amount"
    ]
].sort_values(
    "total_amount",
    ascending=False
).head(10)

display(high_amount)

,order_id,order_status,item_count,product_amount,freight_amount,total_amount
13390,03caa2c082116e1d31e67e9ae3700499,delivered,8.0,13440.0,224.08,13664.08
66599,736e1922ae60d0d6a89247b851902527,delivered,4.0,7160.0,114.88,7274.88
22171,0812eb902a67711a1cb742b3cdaa65ae,delivered,1.0,6735.0,194.31,6929.31
28326,fefacc66af859508bf1a7934eab1e97f,delivered,1.0,6729.0,193.21,6922.21
3508,f5136e38d1a14a4dbd87dff67da82701,delivered,1.0,6499.0,227.66,6726.66
32322,2cc9089445046817a7539d90805e6e5a,delivered,6.0,5934.6,146.94,6081.54
53352,a96610ab360d42a2e5335a3998b4718a,delivered,1.0,4799.0,151.34,4950.34
41086,b4c4b76c642808cbe472a32b86cddc95,canceled,2.0,4599.9,209.54,4809.44
40327,199af31afc78c699f0dbf71fb178d4d4,delivered,1.0,4690.0,74.34,4764.34
40342,8dbc85d1447242f3b127dda390d56e19,delivered,1.0,4590.0,91.78,4681.78


In [77]:
# 第一类：实际送达早于获批
exclude_approval = orders_full.loc[
    valid_model & bad_approval_delivery,
    ["order_id"]
].copy()

exclude_approval["exclusion_reason"] = "delivery_before_approval"


# 第二类：实际送达早于物流交接
exclude_carrier = orders_full.loc[
    valid_model & bad_delivery,
    ["order_id"]
].copy()

exclude_carrier["exclusion_reason"] = "delivery_before_carrier"


# 合并两类异常订单
excluded_detail = pd.concat(
    [exclude_approval, exclude_carrier],
    ignore_index=True
)

# 按订单ID去重，同时保留异常原因
excluded_orders = (
    excluded_detail.groupby("order_id", as_index=False)
    .agg(
        exclusion_reason=(
            "exclusion_reason",
            lambda x: ";".join(sorted(set(x)))
        )
    )
)

# 检查结果
print("送达早于获批：", len(exclude_approval))
print("送达早于物流交接：", len(exclude_carrier))
print("最终排除订单数：", len(excluded_orders))
print("订单ID唯一：", excluded_orders["order_id"].is_unique)

送达早于获批： 61
送达早于物流交接： 23
最终排除订单数： 84
订单ID唯一： True


In [78]:
output_path = (
    project_root
    / "data"
    / "processed"
    / "excluded_orders.csv"
)

excluded_orders.to_csv(
    output_path,
    index=False
)

print("保存成功：", output_path)

保存成功： d:\各类文件\项目\olist-delivery-risk\data\processed\excluded_orders.csv


1